# NorthForge Finance — End-to-End Workflow Run

Runs the full business workflow through `WorkflowOrchestrator`: Foundry's Trial Balance
pipeline (staging → enrichment → reporting → posting → interface) followed by the GL
import of that pipeline's Interface output — all under a single `WorkflowRun`.

Each section below reads and displays the data actually persisted at that stage, straight
from the domain repositories (`TrialBalanceRepository` for Foundry, `GLRepository` for GL).

## Spark session

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master('local[*]')
    .appName('foundry-dev')
    .config(
        'spark.jars.packages',
        'org.postgresql:postgresql:42.7.7',
    )
    .getOrCreate()
)

spark.sparkContext.setLogLevel('ERROR')

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/27 14:16:38 WARN Utils: Your hostname, lionix, resolves to a loopback address: 127.0.1.1; using 192.168.1.7 instead (on interface wlo1)
26/08/27 14:16:38 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/leo/northforge-studio/northforge-finance/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/leo/.ivy2.5.2/cache
The jars for the packages stored in: /home/leo/.ivy2.5.2/jars
org.postgresql#postgresql added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-0d0d0ea4-6bbd-4864-940f-6bdf844a0b4e;1.0
	confs: [default]


	found org.postgresql#postgresql;42.7.7 in central
	found org.checkerframework#checker-qual;3.49.3 in central
:: resolution report :: resolve 60ms :: artifacts dl 2ms
	:: modules in use:
	org.checkerframework#checker-qual;3.49.3 from central in [default]
	org.postgresql#postgresql;42.7.7 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   2   |   0   |   0   |   0   ||   2   |   0   |
	---------------------------------------------------------------------
:: retrieving :: org.apache.spark#spark-submit-parent-0d0d0ea4-6bbd-4864-940f-6bdf844a0b4e
	confs: [default]
	0 artifacts copied, 2 already retrieved (0kB/2ms)
26/08/27 14:16:38 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... 

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/08/27 14:16:39 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


## Imports and helpers

In [2]:
from dataclasses import asdict
from datetime import date

import pandas as pd
from pyspark.sql import functions as F

from foundry.pipeline import TrialBalancePipeline
from foundry.repository import TrialBalanceRepository
from foundry.config.settings import (
    CSV_TABLE_LOCATIONS,
    POSTGRES_TABLE_LOCATIONS
)

from core.store import PostgresStore
from core.runs import (
    RunRepository,
    RunTracker
)

from spec import SpecClient
from atlas import AtlasClient
from reference import ReferenceClient

from registry import RegistryClient
from gl import GLClient
from gl.repository import GLRepository

from workflow import WorkflowOrchestrator


def display_df(df):
    display(df.toPandas())


def display_records(records):
    # GLRepository reads (e.g. get_postings/get_rejections) return tuples
    # of dataclasses rather than Spark DataFrames.
    display(pd.DataFrame([asdict(record) for record in records]))

## Configure clients and build the orchestrator

`TrialBalancePipeline` only knows Foundry processing now — no `RunTracker` — and
`GLClient` only knows GL processing. `WorkflowOrchestrator` is the thin layer that owns
execution/workflow lifecycle across both and coordinates Foundry → GL.

In [3]:
BUSINESS_DT = date(2026, 3, 31)

store = PostgresStore(
    spark,
    table_names = POSTGRES_TABLE_LOCATIONS
)

repository = TrialBalanceRepository(store)

spec = SpecClient.from_db(
    spark,
    transformation_table = 'spec.transformation',
    file_layout_table = 'spec.file_layout'
    
)
atlas = AtlasClient.from_db(
    spark=spark,
    metadata_table='atlas.meta',
    data_table='atlas.data',
)
reference = ReferenceClient.from_db(
    spark = spark,
    fx_rate_table='reference.fx_rate',
    counterparty_table='reference.counterparty',
)

run_repository = RunRepository()
run_tracker = RunTracker(
    repository = run_repository
)

pipeline = TrialBalancePipeline(
    business_dt=BUSINESS_DT,
    repository=repository,
    spec=spec,
    atlas=atlas,
    reference=reference,
)

registry = RegistryClient.from_db(
    spark=spark,
    entity_table='registry.gl_entity',
    department_table='registry.gl_dept',
    branch_table='registry.gl_branch',
    account_table='registry.gl_account',
    sub_account_table='registry.gl_sub_account',
    affiliate_table='registry.gl_affiliate',
    product_table='registry.gl_product',
    book_table='registry.gl_book',
    source_table='registry.gl_source',
)

gl_store = PostgresStore(
    spark,
    table_names={
        'SEGMENT_DEFAULT': 'gl.segment_default',
        'POSTING': 'gl.posting',
        'REJECTION': 'gl.rejection',
        'INTERFACE_TRIAL_BALANCE': 'interface.trial_balance',
    },
)
gl_repository = GLRepository(gl_store, spark)
gl = GLClient(gl_repository, registry)

orchestrator = WorkflowOrchestrator(
    run_tracker=run_tracker,
    foundry_pipeline=pipeline,
    gl=gl,
)

## Run the full workflow (Foundry → GL)

`run_workflow()` creates a single `WorkflowRun`, executes the Foundry pipeline
(`STAGING → ENRICHMENT → REPORTING → POSTING → INTERFACE`), then runs `GL / IMPORT`
against that same workflow's `FOUNDRY / INTERFACE` output — all as one business workflow.

In [4]:
workflow_result = orchestrator.run_workflow()

print(f"Foundry pipeline run complete: {workflow_result.foundry}")
print()
print(f"GL import run complete: {workflow_result.gl}")

business_dt = pipeline.config.business_dt

workflow_run_id = workflow_result.foundry.identity.workflow_run_id
gl_run_id = workflow_result.gl.producer_run_id

# Each Foundry zone's own execution identity (producer_run_id) — this is
# what inter-zone/read-layer selection uses now, not business_dt/batch_id.
staging_zone, enrichment_zone, reporting_zone, posting_zone, interface_zone = (
    workflow_result.foundry.zones
)
staging_producer_run_id = staging_zone.identity.run_id
enrichment_producer_run_id = enrichment_zone.identity.run_id
reporting_producer_run_id = reporting_zone.identity.run_id
posting_producer_run_id = posting_zone.identity.run_id
interface_producer_run_id = interface_zone.identity.run_id

Foundry pipeline run complete: PipelineResult(identity=RunIdentity(workflow_run_id=UUID('4e1cc358-eb41-4ec9-90d8-c7b36e1e794d'), run_id=UUID('7b167733-821f-402d-b411-9cc8eb3891e8'), parent_run_id=None), status=<RunStatus.SUCCEEDED: 'SUCCEEDED'>, zones=(ZoneResult(identity=RunIdentity(workflow_run_id=UUID('4e1cc358-eb41-4ec9-90d8-c7b36e1e794d'), run_id=UUID('62de3807-fe06-4a98-8070-dbcaf1d549e9'), parent_run_id=UUID('7b167733-821f-402d-b411-9cc8eb3891e8')), zone='STAGING', status=<RunStatus.SUCCEEDED: 'SUCCEEDED'>, record_count=42), ZoneResult(identity=RunIdentity(workflow_run_id=UUID('4e1cc358-eb41-4ec9-90d8-c7b36e1e794d'), run_id=UUID('4df667d3-8b0b-40ea-a97a-2ff689814573'), parent_run_id=UUID('7b167733-821f-402d-b411-9cc8eb3891e8')), zone='ENRICHMENT', status=<RunStatus.SUCCEEDED: 'SUCCEEDED'>, record_count=7), ZoneResult(identity=RunIdentity(workflow_run_id=UUID('4e1cc358-eb41-4ec9-90d8-c7b36e1e794d'), run_id=UUID('9c0c1355-793a-4f33-97c5-bb57c3a46f6b'), parent_run_id=UUID('7b167733

## Foundry persistence layers

Each Foundry zone is read straight from its own persisted table, selected by the
`producer_run_id` of the execution that produced it (via `TrialBalanceRepository`) — not
by `business_dt`/`batch_id`.

### Source

In [5]:
src_df = repository.read_source(business_dt=business_dt)

display_df(src_df)

,AS_OF_DT,BUSINESS_DT,SRC_APP_CD,SRC_RECORD_ID,SRC_ENTITY_CD,SRC_BOOKING_DEPT_CD,SRC_ACCOUNT_ID,SRC_ACCT_TYPE,SRC_CLIENT_ID,CPTY_REF_ID,SRC_MEASURE_NM,SRC_MEASURE_CCY_CD,SRC_MEASURE_TRANS_AMT,POSTING_MEASURE_CCY_CD
0,2026-03-31,2026-03-31,NFM,rec-1,USM,TRD,1000,ASSET,,,SRC_PREV_DAY_BAL_AMT,USD,80000.000000000000,USD
1,2026-03-31,2026-03-31,NFM,rec-1,USM,TRD,1000,ASSET,,,SRC_CURRENT_DAY_DEBIT,USD,10000.000000000000,USD
2,2026-03-31,2026-03-31,NFM,rec-1,USM,TRD,1000,ASSET,,,SRC_CURRENT_DAY_CREDIT,USD,0E-12,USD
3,2026-03-31,2026-03-31,NFM,rec-1,USM,TRD,1000,ASSET,,,SRC_CURRENT_DAY_EOD_BALANCE,USD,90000.000000000000,USD
4,2026-03-31,2026-03-31,NFM,rec-1,USM,TRD,1000,ASSET,,,SRC_BACK_VALUED_ADJUSTMENT,USD,0E-12,USD
5,2026-03-31,2026-03-31,NFM,rec-1,USM,TRD,1000,ASSET,,,SRC_ADJUSTED_BALANCE,USD,90000.000000000000,USD
6,2026-03-31,2026-03-31,NFM,rec-2,USM,FIN,1200,ASSET,,,SRC_PREV_DAY_BAL_AMT,USD,15000.000000000000,USD
7,2026-03-31,2026-03-31,NFM,rec-2,USM,FIN,1200,ASSET,,,SRC_CURRENT_DAY_DEBIT,USD,5000.000000000000,USD
8,2026-03-31,2026-03-31,NFM,rec-2,USM,FIN,1200,ASSET,,,SRC_CURRENT_DAY_CREDIT,USD,0E-12,USD
9,2026-03-31,2026-03-31,NFM,rec-2,USM,FIN,1200,ASSET,,,SRC_CURRENT_DAY_EOD_BALANCE,USD,20000.000000000000,USD


### Staging

In [6]:
stg_df = repository.read_staging(staging_producer_run_id)

display_df(stg_df)

,AS_OF_DT,BUSINESS_DT,SRC_APP_CD,DATACLASS,SRC_RECORD_ID,SRC_ENTITY_CD,SRC_BOOKING_DEPT_CD,SRC_ACCOUNT_ID,SRC_ACCT_TYPE,NORM_ACCT_SIGN,...,POSTING_MEASURE_CCY_CD,POSTING_MEASURE_NM,MEASURE_TYPE,POSTING_MEASURE_FUNC_CCY_CD,POSTING_MEASURE_TRANS_AMT,FX_RATE,POSTING_MEASURE_FUNC_AMT,CR_DR_EVALUATOR,WORKFLOW_RUN_ID,PRODUCER_RUN_ID
0,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-1,USM,TRD,1000,ASSET,DEBIT,...,USD,ADJUSTED_BALANCE,POSTABLE,USD,90000.000000000000,1.000000000000,90000.000000000000,DEBIT,4e1cc358-eb41-4ec9-90d8-c7b36e1e794d,62de3807-fe06-4a98-8070-dbcaf1d549e9
1,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-1,USM,TRD,1000,ASSET,DEBIT,...,USD,BACK_VALUE_ADJUSTED_BALANCE,REPORTABLE,USD,0E-12,1.000000000000,0E-12,DEBIT,4e1cc358-eb41-4ec9-90d8-c7b36e1e794d,62de3807-fe06-4a98-8070-dbcaf1d549e9
2,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-1,USM,TRD,1000,ASSET,DEBIT,...,USD,CURRENT_DAY_CREDIT_BALANCE,REPORTABLE,USD,0E-12,1.000000000000,0E-12,DEBIT,4e1cc358-eb41-4ec9-90d8-c7b36e1e794d,62de3807-fe06-4a98-8070-dbcaf1d549e9
3,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-1,USM,TRD,1000,ASSET,DEBIT,...,USD,CURRENT_DAY_DEBIT_BALANCE,REPORTABLE,USD,10000.000000000000,1.000000000000,10000.000000000000,DEBIT,4e1cc358-eb41-4ec9-90d8-c7b36e1e794d,62de3807-fe06-4a98-8070-dbcaf1d549e9
4,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-1,USM,TRD,1000,ASSET,DEBIT,...,USD,CURRENT_DAY_EOD_BALANCE,REPORTABLE,USD,90000.000000000000,1.000000000000,90000.000000000000,DEBIT,4e1cc358-eb41-4ec9-90d8-c7b36e1e794d,62de3807-fe06-4a98-8070-dbcaf1d549e9
5,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-1,USM,TRD,1000,ASSET,DEBIT,...,USD,PREVIOUS_DAY_BALANCE,REPORTABLE,USD,80000.000000000000,1.000000000000,80000.000000000000,DEBIT,4e1cc358-eb41-4ec9-90d8-c7b36e1e794d,62de3807-fe06-4a98-8070-dbcaf1d549e9
6,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-2,USM,FIN,1200,ASSET,DEBIT,...,USD,ADJUSTED_BALANCE,POSTABLE,USD,20000.000000000000,1.000000000000,20000.000000000000,DEBIT,4e1cc358-eb41-4ec9-90d8-c7b36e1e794d,62de3807-fe06-4a98-8070-dbcaf1d549e9
7,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-2,USM,FIN,1200,ASSET,DEBIT,...,USD,BACK_VALUE_ADJUSTED_BALANCE,REPORTABLE,USD,0E-12,1.000000000000,0E-12,DEBIT,4e1cc358-eb41-4ec9-90d8-c7b36e1e794d,62de3807-fe06-4a98-8070-dbcaf1d549e9
8,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-2,USM,FIN,1200,ASSET,DEBIT,...,USD,CURRENT_DAY_CREDIT_BALANCE,REPORTABLE,USD,0E-12,1.000000000000,0E-12,DEBIT,4e1cc358-eb41-4ec9-90d8-c7b36e1e794d,62de3807-fe06-4a98-8070-dbcaf1d549e9
9,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-2,USM,FIN,1200,ASSET,DEBIT,...,USD,CURRENT_DAY_DEBIT_BALANCE,REPORTABLE,USD,5000.000000000000,1.000000000000,5000.000000000000,DEBIT,4e1cc358-eb41-4ec9-90d8-c7b36e1e794d,62de3807-fe06-4a98-8070-dbcaf1d549e9


### Enrichment

In [7]:
enr_df = repository.read_enrichment(enrichment_producer_run_id)

display_df(enr_df)

,AS_OF_DT,BUSINESS_DT,SRC_APP_CD,DATACLASS,SRC_RECORD_ID,SRC_ENTITY_CD,SRC_BOOKING_DEPT_CD,SRC_ACCOUNT_ID,SRC_ACCT_TYPE,NORM_ACCT_SIGN,...,GL_ACCOUNT_DR,GL_ACCOUNT_CR,GL_ACCOUNT,GL_SUB_ACCOUNT,GL_AFFILIATE_CD,GL_PRODUCT_CD,GL_BOOK_CD,GL_COA_SRC_SEGMENT,WORKFLOW_RUN_ID,PRODUCER_RUN_ID
0,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-1,USM,TRD,1000,ASSET,DEBIT,...,101000,201000,101000,1000,000000,000000,LOCAL_GAAP,NFM_TB,4e1cc358-eb41-4ec9-90d8-c7b36e1e794d,4df667d3-8b0b-40ea-a97a-2ff689814573
1,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-2,USM,FIN,1200,ASSET,DEBIT,...,120000,220000,120000,1200,000000,000000,LOCAL_GAAP,NFM_TB,4e1cc358-eb41-4ec9-90d8-c7b36e1e794d,4df667d3-8b0b-40ea-a97a-2ff689814573
2,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-3,USM,TRD,2000,LIABILITY,CREDIT,...,130000,210000,210000,2000,000000,000000,LOCAL_GAAP,NFM_TB,4e1cc358-eb41-4ec9-90d8-c7b36e1e794d,4df667d3-8b0b-40ea-a97a-2ff689814573
3,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-4,USM,FIN,4000,REVENUE,CREDIT,...,410000,410000,410000,4000,000000,000000,LOCAL_GAAP,NFM_TB,4e1cc358-eb41-4ec9-90d8-c7b36e1e794d,4df667d3-8b0b-40ea-a97a-2ff689814573
4,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-5,USM,FIN,3000,EQUITY,CREDIT,...,310000,310000,310000,3000,000000,000000,LOCAL_GAAP,NFM_TB,4e1cc358-eb41-4ec9-90d8-c7b36e1e794d,4df667d3-8b0b-40ea-a97a-2ff689814573
5,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-6,CAM,TRD,1000,ASSET,DEBIT,...,101000,201000,101000,1000,000000,000000,LOCAL_GAAP,NFM_TB,4e1cc358-eb41-4ec9-90d8-c7b36e1e794d,4df667d3-8b0b-40ea-a97a-2ff689814573
6,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-7,CAM,TRD,2100,LIABILITY,CREDIT,...,140000,230000,230000,2100,1000,000000,LOCAL_GAAP,NFM_TB,4e1cc358-eb41-4ec9-90d8-c7b36e1e794d,4df667d3-8b0b-40ea-a97a-2ff689814573


### Reporting

In [8]:
rpt_df = repository.read_reporting(reporting_producer_run_id)

display_df(rpt_df)

,AS_OF_DT,BUSINESS_DT,SRC_APP_CD,DATACLASS,SRC_RECORD_ID,SRC_ENTITY_CD,SRC_BOOKING_DEPT_CD,SRC_ACCOUNT_ID,SRC_ACCT_TYPE,NORM_ACCT_SIGN,...,GL_ACCOUNT_DR,GL_ACCOUNT_CR,GL_ACCOUNT,GL_SUB_ACCOUNT,GL_AFFILIATE_CD,GL_PRODUCT_CD,GL_BOOK_CD,GL_COA_SRC_SEGMENT,WORKFLOW_RUN_ID,PRODUCER_RUN_ID
0,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-1,USM,TRD,1000,ASSET,DEBIT,...,101000,201000,101000,1000,000000,000000,LOCAL_GAAP,NFM_TB,4e1cc358-eb41-4ec9-90d8-c7b36e1e794d,9c0c1355-793a-4f33-97c5-bb57c3a46f6b
1,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-1,USM,TRD,1000,ASSET,DEBIT,...,,,,,,,,,4e1cc358-eb41-4ec9-90d8-c7b36e1e794d,9c0c1355-793a-4f33-97c5-bb57c3a46f6b
2,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-1,USM,TRD,1000,ASSET,DEBIT,...,,,,,,,,,4e1cc358-eb41-4ec9-90d8-c7b36e1e794d,9c0c1355-793a-4f33-97c5-bb57c3a46f6b
3,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-1,USM,TRD,1000,ASSET,DEBIT,...,,,,,,,,,4e1cc358-eb41-4ec9-90d8-c7b36e1e794d,9c0c1355-793a-4f33-97c5-bb57c3a46f6b
4,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-1,USM,TRD,1000,ASSET,DEBIT,...,,,,,,,,,4e1cc358-eb41-4ec9-90d8-c7b36e1e794d,9c0c1355-793a-4f33-97c5-bb57c3a46f6b
5,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-1,USM,TRD,1000,ASSET,DEBIT,...,,,,,,,,,4e1cc358-eb41-4ec9-90d8-c7b36e1e794d,9c0c1355-793a-4f33-97c5-bb57c3a46f6b
6,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-2,USM,FIN,1200,ASSET,DEBIT,...,120000,220000,120000,1200,000000,000000,LOCAL_GAAP,NFM_TB,4e1cc358-eb41-4ec9-90d8-c7b36e1e794d,9c0c1355-793a-4f33-97c5-bb57c3a46f6b
7,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-2,USM,FIN,1200,ASSET,DEBIT,...,,,,,,,,,4e1cc358-eb41-4ec9-90d8-c7b36e1e794d,9c0c1355-793a-4f33-97c5-bb57c3a46f6b
8,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-2,USM,FIN,1200,ASSET,DEBIT,...,,,,,,,,,4e1cc358-eb41-4ec9-90d8-c7b36e1e794d,9c0c1355-793a-4f33-97c5-bb57c3a46f6b
9,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-2,USM,FIN,1200,ASSET,DEBIT,...,,,,,,,,,4e1cc358-eb41-4ec9-90d8-c7b36e1e794d,9c0c1355-793a-4f33-97c5-bb57c3a46f6b


### Posting

In [9]:
pst_df = repository.read_posting(posting_producer_run_id)

display_df(pst_df)

,AS_OF_DT,BUSINESS_DT,SRC_APP_CD,DATACLASS,SRC_RECORD_ID,POSTING_ID,SRC_ENTITY_CD,SRC_BOOKING_DEPT_CD,SRC_ACCOUNT_ID,SRC_ACCT_TYPE,...,BACK_VALUE_ADJUSTED_BALANCE,ADJUSTED_BALANCE,POSTING_PREVIOUS_DAY_BALANCE,POSTING_CURRENT_DAY_DEBIT_BALANCE,POSTING_CURRENT_DAY_CREDIT_BALANCE,POSTING_CURRENT_DAY_EOD_BALANCE,POSTING_BACK_VALUE_ADJUSTED_BALANCE,POSTING_ADJUSTED_BALANCE,WORKFLOW_RUN_ID,PRODUCER_RUN_ID
0,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-1,PST-260331-260331-79f22f87-8d80-4ddc-bd7a-9975...,USM,TRD,1000,ASSET,...,0E-12,90000.000000000000,80000.000000000000,10000.000000000000,0E-12,90000.000000000000,0E-12,90000.000000000000,4e1cc358-eb41-4ec9-90d8-c7b36e1e794d,79f22f87-8d80-4ddc-bd7a-9975fecbe1b0
1,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-2,PST-260331-260331-79f22f87-8d80-4ddc-bd7a-9975...,USM,FIN,1200,ASSET,...,0E-12,20000.000000000000,15000.000000000000,5000.000000000000,0E-12,20000.000000000000,0E-12,20000.000000000000,4e1cc358-eb41-4ec9-90d8-c7b36e1e794d,79f22f87-8d80-4ddc-bd7a-9975fecbe1b0
2,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-3,PST-260331-260331-79f22f87-8d80-4ddc-bd7a-9975...,USM,TRD,2000,LIABILITY,...,0E-12,-65000.000000000000,-55000.000000000000,0E-12,-10000.000000000000,-65000.000000000000,0E-12,-65000.000000000000,4e1cc358-eb41-4ec9-90d8-c7b36e1e794d,79f22f87-8d80-4ddc-bd7a-9975fecbe1b0
3,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-4,PST-260331-260331-79f22f87-8d80-4ddc-bd7a-9975...,USM,FIN,4000,REVENUE,...,0E-12,-25000.000000000000,-20000.000000000000,0E-12,-5000.000000000000,-25000.000000000000,0E-12,-25000.000000000000,4e1cc358-eb41-4ec9-90d8-c7b36e1e794d,79f22f87-8d80-4ddc-bd7a-9975fecbe1b0
4,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-5,PST-260331-260331-79f22f87-8d80-4ddc-bd7a-9975...,USM,FIN,3000,EQUITY,...,0E-12,-20000.000000000000,-20000.000000000000,0E-12,0E-12,-20000.000000000000,0E-12,-20000.000000000000,4e1cc358-eb41-4ec9-90d8-c7b36e1e794d,79f22f87-8d80-4ddc-bd7a-9975fecbe1b0
5,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-6,PST-260331-260331-79f22f87-8d80-4ddc-bd7a-9975...,CAM,TRD,1000,ASSET,...,1000.000000000000,51000.000000000000,40000.000000000000,10000.000000000000,0E-12,50000.000000000000,1000.000000000000,51000.000000000000,4e1cc358-eb41-4ec9-90d8-c7b36e1e794d,79f22f87-8d80-4ddc-bd7a-9975fecbe1b0
6,2026-03-31,2026-03-31,NFM,TRIAL_BALANCE,rec-7,PST-260331-260331-79f22f87-8d80-4ddc-bd7a-9975...,CAM,TRD,2100,LIABILITY,...,-1000.000000000000,-51000.000000000000,-40000.000000000000,0E-12,-10000.000000000000,-50000.000000000000,-1000.000000000000,-51000.000000000000,4e1cc358-eb41-4ec9-90d8-c7b36e1e794d,79f22f87-8d80-4ddc-bd7a-9975fecbe1b0


### Interface

This is the Foundry Interface output — the same rows GL reads as input for `GL / IMPORT`,
selected by `WORKFLOW_RUN_ID` + `PRODUCER_RUN_ID` lineage rather than business date/batch.

In [10]:
int_df = repository.read_interface(interface_producer_run_id)

display_df(int_df)

,WORKFLOW_RUN_ID,PRODUCER_RUN_ID,DATACLASS,TRANSACTION_NUMBER,LINE_NUMBER,ENTITY_CD,DEPT_CD,BRANCH_CD,GL_ACCOUNT,SUB_ACCOUNT,...,POSTING_STREAM,SRC_RECORD_ID,SRC_APP_CD,TRANSACTION_CURRENCY,TRANSACTION_AMOUNT,ACCOUNTED_CURRENCY,ACCOUNTED_AMOUNT,FX_RATE,AS_OF_DATE,BUSINESS_DATE
0,4e1cc358-eb41-4ec9-90d8-c7b36e1e794d,d9608958-5780-42b0-a327-3d63774b7317,TRIAL_BALANCE,NFTB-4df667d3-8b0b-40ea-a97a-2ff689814573-USM-...,1,1000,1100,NYC,101000,1000,...,GROSS_UP,rec-1,NFM,USD,90000.000000000000,USD,90000.000000000000,1.000000000000,2026-03-31,2026-03-31
1,4e1cc358-eb41-4ec9-90d8-c7b36e1e794d,d9608958-5780-42b0-a327-3d63774b7317,TRIAL_BALANCE,NFTB-4df667d3-8b0b-40ea-a97a-2ff689814573-USM-...,2,1000,1200,NYC,120000,1200,...,GROSS_UP,rec-2,NFM,USD,20000.000000000000,USD,20000.000000000000,1.000000000000,2026-03-31,2026-03-31
2,4e1cc358-eb41-4ec9-90d8-c7b36e1e794d,d9608958-5780-42b0-a327-3d63774b7317,TRIAL_BALANCE,NFTB-4df667d3-8b0b-40ea-a97a-2ff689814573-USM-...,3,1000,1100,NYC,210000,2000,...,GROSS_UP,rec-3,NFM,USD,-65000.000000000000,USD,-65000.000000000000,1.000000000000,2026-03-31,2026-03-31
3,4e1cc358-eb41-4ec9-90d8-c7b36e1e794d,d9608958-5780-42b0-a327-3d63774b7317,TRIAL_BALANCE,NFTB-4df667d3-8b0b-40ea-a97a-2ff689814573-USM-...,4,1000,1200,NYC,410000,4000,...,GROSS_UP,rec-4,NFM,USD,-25000.000000000000,USD,-25000.000000000000,1.000000000000,2026-03-31,2026-03-31
4,4e1cc358-eb41-4ec9-90d8-c7b36e1e794d,d9608958-5780-42b0-a327-3d63774b7317,TRIAL_BALANCE,NFTB-4df667d3-8b0b-40ea-a97a-2ff689814573-USM-...,5,1000,1200,NYC,310000,3000,...,GROSS_UP,rec-5,NFM,USD,-20000.000000000000,USD,-20000.000000000000,1.000000000000,2026-03-31,2026-03-31
5,4e1cc358-eb41-4ec9-90d8-c7b36e1e794d,d9608958-5780-42b0-a327-3d63774b7317,TRIAL_BALANCE,NFTB-4df667d3-8b0b-40ea-a97a-2ff689814573-CAM-...,1,2000,2100,TOR,101000,1000,...,GROSS_UP,rec-6,NFM,CAD,51000.000000000000,USD,38250.000000000000,0.750000000000,2026-03-31,2026-03-31
6,4e1cc358-eb41-4ec9-90d8-c7b36e1e794d,d9608958-5780-42b0-a327-3d63774b7317,TRIAL_BALANCE,NFTB-4df667d3-8b0b-40ea-a97a-2ff689814573-CAM-...,2,2000,2100,TOR,230000,2100,...,GROSS_UP,rec-7,NFM,CAD,-51000.000000000000,USD,-38250.000000000000,0.750000000000,2026-03-31,2026-03-31


## GL persistence layer

`GL / IMPORT` (`gl_run_id`) reads the Interface rows produced by `FOUNDRY / INTERFACE`
above and stamps its own execution identity onto whatever it writes — `gl.posting` and
`gl.rejection` rows carry `PRODUCER_RUN_ID = gl_run_id`, not the Interface producer's ID.

### GL Posting

In [11]:
gl_postings = gl.get_postings(gl_run_id)

display_records(gl_postings)

,gl_posting_id,posted_at,workflow_run_id,producer_run_id,dataclass,transaction_number,line_number,foundry_rule_id,posting_id,posting_stream,...,book_cd,source_cd,cr_dr_ind,transaction_currency,transaction_amount,accounted_currency,accounted_amount,fx_rate,as_of_date,business_date
0,91d39dca-285c-4537-84e1-56c062b670e2,2026-08-27 14:16:58.300266,4e1cc358-eb41-4ec9-90d8-c7b36e1e794d,82b3fb95-93b2-491a-8c42-54f252cdbcca,TRIAL_BALANCE,NFTB-4df667d3-8b0b-40ea-a97a-2ff689814573-CAM-...,1,TB-GROSS-UP,PST-260331-260331-79f22f87-8d80-4ddc-bd7a-9975...,GROSS_UP,...,LOCAL_GAAP,NFM_TB,DR,CAD,51000.000000000000,USD,38250.000000000000,0.750000000000,2026-03-31,2026-03-31
1,209d5ce3-5cc5-4fe3-8997-4b611483fb2d,2026-08-27 14:16:59.812919,4e1cc358-eb41-4ec9-90d8-c7b36e1e794d,82b3fb95-93b2-491a-8c42-54f252cdbcca,TRIAL_BALANCE,NFTB-4df667d3-8b0b-40ea-a97a-2ff689814573-CAM-...,2,TB-GROSS-UP,PST-260331-260331-79f22f87-8d80-4ddc-bd7a-9975...,GROSS_UP,...,LOCAL_GAAP,NFM_TB,CR,CAD,-51000.000000000000,USD,-38250.000000000000,0.750000000000,2026-03-31,2026-03-31
2,facd438a-d452-4106-b8d6-3a88c7a13c9d,2026-08-27 14:17:00.627763,4e1cc358-eb41-4ec9-90d8-c7b36e1e794d,82b3fb95-93b2-491a-8c42-54f252cdbcca,TRIAL_BALANCE,NFTB-4df667d3-8b0b-40ea-a97a-2ff689814573-USM-...,1,TB-GROSS-UP,PST-260331-260331-79f22f87-8d80-4ddc-bd7a-9975...,GROSS_UP,...,LOCAL_GAAP,NFM_TB,DR,USD,90000.000000000000,USD,90000.000000000000,1.000000000000,2026-03-31,2026-03-31
3,05ecd006-9cbc-4bc1-8fd8-19ca38a1780c,2026-08-27 14:17:01.380935,4e1cc358-eb41-4ec9-90d8-c7b36e1e794d,82b3fb95-93b2-491a-8c42-54f252cdbcca,TRIAL_BALANCE,NFTB-4df667d3-8b0b-40ea-a97a-2ff689814573-USM-...,2,TB-GROSS-UP,PST-260331-260331-79f22f87-8d80-4ddc-bd7a-9975...,GROSS_UP,...,LOCAL_GAAP,NFM_TB,DR,USD,20000.000000000000,USD,20000.000000000000,1.000000000000,2026-03-31,2026-03-31
4,3db52f0c-0646-4f90-9f90-ba38b6cae2ca,2026-08-27 14:17:02.077794,4e1cc358-eb41-4ec9-90d8-c7b36e1e794d,82b3fb95-93b2-491a-8c42-54f252cdbcca,TRIAL_BALANCE,NFTB-4df667d3-8b0b-40ea-a97a-2ff689814573-USM-...,3,TB-GROSS-UP,PST-260331-260331-79f22f87-8d80-4ddc-bd7a-9975...,GROSS_UP,...,LOCAL_GAAP,NFM_TB,CR,USD,-65000.000000000000,USD,-65000.000000000000,1.000000000000,2026-03-31,2026-03-31
5,33d58bdf-dc4c-418c-a52e-d99cc1552a49,2026-08-27 14:17:02.748087,4e1cc358-eb41-4ec9-90d8-c7b36e1e794d,82b3fb95-93b2-491a-8c42-54f252cdbcca,TRIAL_BALANCE,NFTB-4df667d3-8b0b-40ea-a97a-2ff689814573-USM-...,4,TB-GROSS-UP,PST-260331-260331-79f22f87-8d80-4ddc-bd7a-9975...,GROSS_UP,...,LOCAL_GAAP,NFM_TB,CR,USD,-25000.000000000000,USD,-25000.000000000000,1.000000000000,2026-03-31,2026-03-31
6,a7dc5842-5974-43d3-9474-3133dd5e22bb,2026-08-27 14:17:03.395733,4e1cc358-eb41-4ec9-90d8-c7b36e1e794d,82b3fb95-93b2-491a-8c42-54f252cdbcca,TRIAL_BALANCE,NFTB-4df667d3-8b0b-40ea-a97a-2ff689814573-USM-...,5,TB-GROSS-UP,PST-260331-260331-79f22f87-8d80-4ddc-bd7a-9975...,GROSS_UP,...,LOCAL_GAAP,NFM_TB,CR,USD,-20000.000000000000,USD,-20000.000000000000,1.000000000000,2026-03-31,2026-03-31


### GL Rejection

A non-zero rejected count here is a normal business outcome, not an execution failure —
`GL / IMPORT` still completes as `SUCCEEDED` as long as processing itself ran cleanly.

In [12]:
gl_rejections = gl.get_rejections(gl_run_id)

display_records(gl_rejections)

""


In [13]:
# spark.stop()